# WJX Template Debug

用于调试 `codeyun` 的问卷星模板链路，重点观察：

- 当前配置的问卷星账号和执行问卷
- 登录后的设计页运行状态
- 进入正式编辑页后的 `所属课程` 选项状态
- 新增/隐藏前的安全预检，避免重复新增


In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'backend').exists():
    for candidate in [ROOT, *ROOT.parents]:
        if (candidate / 'backend').exists():
            ROOT = candidate
            break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from sqlmodel import Session
from backend.db import engine
from backend.core.attendance_service import (
    decrypt_attendance_secret,
    get_current_account,
    get_or_create_attendance_service_config,
)
from backend.core.attendance_wjx import (
    _run_state,
    apply_template_changes,
    create_wjx_session,
    ensure_logged_in,
    open_design_page,
    open_edit_page,
    read_course_options,
)

ACTIVITY_ID = '264266843'
QUESTION_TITLE = '所属课程'


In [ ]:
with Session(engine) as db:
    config = get_or_create_attendance_service_config(db)
    account = get_current_account(db, config)
    username = account.login_username
    password = decrypt_attendance_secret(account.password_encrypted)

{
    'activity_id': ACTIVITY_ID,
    'question_title': QUESTION_TITLE,
    'username': username,
    'execution_device_entry_id': config.execution_device_entry_id,
}


In [ ]:
session = create_wjx_session()
ensure_logged_in(
    session,
    username=username,
    password=password,
    target_url=f'https://www.wjx.cn/wjx/design/designstart.aspx?activity={ACTIVITY_ID}',
)
open_design_page(session, activity_id=ACTIVITY_ID)

{
    'url': session.tab.url,
    'run_state': _run_state(session.tab),
}


In [ ]:
open_edit_page(session, activity_id=ACTIVITY_ID, question_title=QUESTION_TITLE)
data = read_course_options(session, question_title=QUESTION_TITLE)
all_names = [item['name'] for item in data['all_items']]

{
    'url': session.tab.url,
    'all_count': len(data['all_items']),
    'visible_count': len(data['visible_names']),
    'visible_names': data['visible_names'],
}


In [ ]:
tail_count = 20
{
    'tail_names': all_names[-tail_count:],
    'placeholder_names': [name for name in all_names if name.startswith('选项')],
}


In [ ]:
hide_names = [
    '20260301第38届念住',
    '20260309梵呗初阶',
    '20251130禅宗7期4.5阶',
]
add_names = [
    '20260401第39届念住',
    '20260401第45届觉观',
]

preview = {
    'hide_now_visible': [name for name in hide_names if name in data['visible_names']],
    'hide_already_hidden': [name for name in hide_names if name not in data['visible_names']],
    'add_already_exists': [name for name in add_names if name in all_names],
    'safe_add': [name for name in add_names if name not in all_names],
}
preview


In [ ]:
# 手动执行时只使用 preview['safe_add']，避免重复新增
# result = apply_template_changes(
#     login_username=username,
#     password=password,
#     activity_id=ACTIVITY_ID,
#     hide_names=hide_names,
#     add_names=preview['safe_add'],
# )
# result


In [ ]:
# 调试完成后记得关浏览器
# session.close()
